# Basic 03 - Plot and Export
Small, step-by-step plotting/export flow with explicit file-load metadata.

Run the import and dataset-loading cells first. Each following example selects what it needs independently; change its study, assay, or run to explore your data.


## Imports and dataset path


In [ ]:
from pathlib import Path

from isa_phm import ISAWrapper

import pandas as pd
from IPython.display import display
from bokeh.io import output_notebook
from bokeh.plotting import show

output_notebook()

# Set this to your dataset's ISA-PHM JSON file.
ISA_JSON = Path("path/to/your-dataset.isa-phm.json").expanduser()
DATA_ROOT = ISA_JSON.parent

print("ISA-JSON :", ISA_JSON)
print("DATA_ROOT:", DATA_ROOT)
print("Exists   :", ISA_JSON.exists())


## Load the wrapper


In [ ]:
wrapper = ISAWrapper(
    ISA_JSON,
    data_root=DATA_ROOT,
    strict_validation=False,
    enable_chunked_large_file_mode=True,
    large_file_threshold_mb=64.0,
    chunk_rows=250000,
)


## Pick first study


In [ ]:
study = wrapper.study(1)
study.title


## Pick first assay


In [ ]:
study = wrapper.study(1)

assay = study.assay(1)
assay.assay_id


## Pick first run


In [ ]:
study = wrapper.study(1)
assay = study.assay(1)

run = assay.run(1)  # Or select by run ID
run.run_id


## Load one run with file metadata


In [ ]:
study = wrapper.study(1)
assay = study.assay(1)
run = assay.run(1)

df, meta = run.load_dataframe_with_meta(file_type="auto")


## Show resolved load metadata


In [ ]:
study = wrapper.study(1)
assay = study.assay(1)
run = assay.run(1)
df, meta = run.load_dataframe_with_meta(file_type="auto")
display(pd.DataFrame([meta.model_dump()]))


## Quick data preview


In [ ]:
study = wrapper.study(1)
assay = study.assay(1)
run = assay.run(1)
df, meta = run.load_dataframe_with_meta(file_type="auto")

display(df.head(10))


## Timeseries plot


In [ ]:
study = wrapper.study(1)
assay = study.assay(1)
run = assay.run(1)

fig_ts = run.plot_timeseries(file_type="auto")
show(fig_ts)


## Frequency-domain plot


In [ ]:
study = wrapper.study(1)
assay = study.assay(1)
run = assay.run(1)

fig_fft = run.plot_frequency_domain(file_type="auto")
show(fig_fft)


## Lifecycle features for this assay


In [ ]:
study = wrapper.study(1)
assay = study.assay(1)

lc = assay.lifecycle_features(file_type="auto")
display(lc.head(10))


## Multi-sensor merged dataframe


In [ ]:
study = wrapper.study(1)

run_id = study.assay(1).run(1).run_id
multi_df = study.load_multi_sensor_dataframe(run_id=run_id, file_type="auto")
display(multi_df.head(10))


## Labeled export dataframe


In [ ]:
study = wrapper.study(1)

labeled_df = study.export_labeled_dataset(file_type="auto")


## Labeled export preview


In [ ]:
study = wrapper.study(1)
labeled_df = study.export_labeled_dataset(file_type="auto")
display(labeled_df.head(10))
display(pd.DataFrame([{"rows": int(labeled_df.shape[0]), "cols": int(labeled_df.shape[1])}]))


## Validation summary before downstream export


In [ ]:
validation = wrapper.validate_dataset(check_files=True)
display(pd.DataFrame([{
    "ok": validation.ok,
    "n_errors": validation.n_errors,
    "n_warnings": validation.n_warnings,
    "n_info": validation.n_info,
}]))


## AI context export (metadata-only)


In [ ]:
ai_ctx = wrapper.ai_context(include_validation=True)
list(ai_ctx.keys())
